**Goal of this notebook:**
1. Define the shared `MIRAState` that flows through all agents
2. Build each of the 4 specialized agent nodes
3. Wire them into a LangGraph `StateGraph` with conditional edges
4. Run a full end-to-end clinical query through the pipeline

```
         [Clinical Question]
                 ↓
   ┌─────────────────────────────┐
   │  Agent 1: SQL Data Extractor │ ←── retries if SQL fails
   └─────────────────────────────┘
                 ↓
   ┌─────────────────────────────┐
   │  Agent 2: Semantic Cross-Ref │ ←── queries FAISS
   └─────────────────────────────┘
                 ↓
   ┌─────────────────────────────┐
   │  Agent 3: Clinical Reasoning │ ←── synthesizes SQL + RAG
   └─────────────────────────────┘
                 ↓
   ┌─────────────────────────────┐
   │  Agent 4: Critic & Safety   │ ←── validates, then END
   └─────────────────────────────┘
```

In [ ]:
import os
import sqlite3
import pickle
import json
import numpy as np
import pandas as pd
import faiss
from pathlib import Path
from typing import TypedDict, Annotated
from openai import OpenAI

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

OPENAI_API_KEY=""
DB_PATH        = Path("./mira_data/mimic.db")
FAISS_PATH     = Path("./mira_data/medical_faiss.index")
META_PATH      = Path("./mira_data/faiss_metadata.pkl")
SCHEMA_PATH    = Path("./mira_data/db_schema.txt")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

print("✅ Imports done")

✅ Imports done


In [13]:
# SQLite
conn = sqlite3.connect(DB_PATH)
DB_SCHEMA = SCHEMA_PATH.read_text()
print("✅ SQLite connected")

# FAISS
index = faiss.read_index(str(FAISS_PATH))
with open(META_PATH, "rb") as f:
    MEDICAL_GUIDELINES = pickle.load(f)
print(f"✅ FAISS loaded: {index.ntotal} vectors")

# OpenAI embedding helper
def get_openai_embeddings(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([r.embedding for r in response.data], dtype=np.float32)

print("✅ Data layer ready")

✅ SQLite connected
✅ FAISS loaded: 11 vectors
✅ Data layer ready


In [14]:
# defining tools which can be used by the agent , just as we did in the previous notebook
@tool
def sql_query(query: str) -> str:
    """
    Execute a SQL SELECT query against the MIMIC-IV patient database.
    Use this to retrieve live patient data: demographics, admissions, lab results.
    Tables: patients, admissions, labevents, d_labitems, diagnoses_icd.
    Always JOIN d_labitems ON labevents.itemid = d_labitems.itemid for readable lab names.
    Filter abnormal labs with: WHERE labevents.flag = 'abnormal'
    Returns JSON string of results or error.
    """
    try:
        df = pd.read_sql_query(query, conn)
        if df.empty:
            return json.dumps({"result": "No data found."})
        return json.dumps({"rows": df.head(20).to_dict(orient="records")}, default=str)
    except Exception as e:
        return json.dumps({"error": str(e), "hint": "Check table/column names against the schema."})


@tool
def vector_search(query: str, k: int = 3) -> str:
    """
    Search the medical knowledge base using semantic similarity.
    Use this to find clinical guidelines, treatment protocols, diagnostic criteria.
    Input: a clinical description or condition name.
    Returns top-k most relevant guideline chunks with source citations.
    """
    try:
        query_vec = get_openai_embeddings([query])
        distances, indices = index.search(query_vec, k)
        results = []
        for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            chunk = MEDICAL_GUIDELINES[idx].copy()
            chunk["rank"] = rank + 1
            chunk["relevance_score"] = round(1 / (1 + float(dist)), 4)
            results.append(chunk)
        return json.dumps({"guidelines": results}, default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})


MIRA_TOOLS = [sql_query, vector_search]
print("✅ Tools ready")

✅ Tools ready


In [15]:
# This is the important cell:
# In this cell we are defining the state of the agent, which will be used to store the intermediate results of each agent in the pipeline.

class MIRAState(TypedDict):
    # Input
    clinical_question: str          # original user question

    # Agent 1 output
    sql_query_used: str             # the SQL query Agent 1 wrote
    sql_result: str                 # raw JSON result from SQLite
    sql_retry_count: int            # how many times Agent 1 retried
    sql_error: str                  # error message if SQL failed

    # Agent 2 output
    search_query_used: str          # the semantic query Agent 2 used
    guidelines: str                 # raw JSON result from FAISS

    # Agent 3 output
    clinical_reasoning: str         # synthesized analysis

    # Agent 4 output
    final_report: str               # validated, safe final response
    safety_flags: list[str]         # any safety concerns flagged
    approved: bool                  # True = safe to show user


print("✅ MIRAState defined")
print("   Fields:", list(MIRAState.__annotations__.keys()))

✅ MIRAState defined
   Fields: ['clinical_question', 'sql_query_used', 'sql_result', 'sql_retry_count', 'sql_error', 'search_query_used', 'guidelines', 'clinical_reasoning', 'final_report', 'safety_flags', 'approved']


In [16]:
# here we have defined the first agent which is the SQL extractor agent , this agent will take the clinical question and generate a sql query to extract the relevant data from database.

def agent1_sql_extractor(state: MIRAState) -> MIRAState:
    print("\n" + "="*60)
    print("🤖 AGENT 1: SQL Data Extractor")
    print("="*60)

    retry_count = state.get("sql_retry_count", 0)
    previous_error = state.get("sql_error", "")

    # Build prompt — include error context if retrying
    error_context = ""
    if previous_error:
        error_context = f"""
Your previous SQL query failed with this error:
{previous_error}

Fix the query based on this error and the schema above.
"""

    system_prompt = f"""You are a medical SQL expert. Write a single valid SQLite SELECT query to answer the clinical question.

DATABASE SCHEMA:
{DB_SCHEMA}

RULES:
- Write only valid SQLite SQL
- Always JOIN d_labitems ON labevents.itemid = d_labitems.itemid to get lab names
- Use WHERE labevents.flag = 'abnormal' for abnormal results
- Return only the raw SQL query, nothing else, no markdown, no explanation
{error_context}"""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Clinical question: {state['clinical_question']}")
    ])

    # Clean up the SQL (remove markdown fences if present)
    raw_sql = response.content.strip()
    raw_sql = raw_sql.replace("```sql", "").replace("```", "").strip()

    print(f"📝 SQL Generated:\n{raw_sql}")

    # Execute
    result = sql_query.invoke({"query": raw_sql})
    parsed = json.loads(result)

    if "error" in parsed:
        print(f"❌ SQL Error: {parsed['error']}")
        return {
            **state,
            "sql_query_used": raw_sql,
            "sql_result": "",
            "sql_error": parsed["error"],
            "sql_retry_count": retry_count + 1
        }

    rows = parsed.get("rows", [])
    print(f"✅ SQL Success: {len(rows)} rows returned")
    return {
        **state,
        "sql_query_used": raw_sql,
        "sql_result": result,
        "sql_error": "",
        "sql_retry_count": retry_count
    }


def should_retry_sql(state: MIRAState) -> str:
    """
    Conditional edge after Agent 1.
    If SQL failed AND we haven't exceeded max retries → retry Agent 1.
    Otherwise → proceed to Agent 2.
    """
    if state.get("sql_error") and state.get("sql_retry_count", 0) < 3:
        print(f"🔄 Retrying SQL (attempt {state['sql_retry_count'] + 1}/3)...")
        return "retry"
    return "ok"


print("✅ Agent 1 defined")

✅ Agent 1 defined


**Job:** Look at what Agent 1 found (the patient's lab values) → form a semantic query → search FAISS → return relevant clinical guidelines.

In [17]:
def agent2_semantic_crossref(state: MIRAState) -> MIRAState:
    print("\n" + "="*60)
    print("🤖 AGENT 2: Semantic Cross-Ref")
    print("="*60)

    # If Agent 1 failed entirely, search based on the original question
    sql_context = state.get("sql_result", "") or "No patient data retrieved."

    system_prompt = """You are a medical research assistant. 
Based on the patient data and clinical question below, write a concise semantic search query (1-2 sentences) 
to find the most relevant clinical guidelines. Focus on the specific abnormal values or conditions found.
Return ONLY the search query string, nothing else."""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
Clinical question: {state['clinical_question']}

Patient data found:
{sql_context[:1000]}

Write a semantic search query to find relevant guidelines:
""")
    ])

    search_query = response.content.strip()
    print(f"🔍 Semantic Query: {search_query}")

    # Search FAISS
    guidelines_result = vector_search.invoke({"query": search_query, "k": 3})
    parsed = json.loads(guidelines_result)

    print(f"✅ Found {len(parsed.get('guidelines', []))} guideline chunks")
    for g in parsed.get("guidelines", []):
        print(f"   #{g['rank']} [{g['source']}] — {g['topic']}")

    return {
        **state,
        "search_query_used": search_query,
        "guidelines": guidelines_result
    }


print("✅ Agent 2 defined")

✅ Agent 2 defined


**Job:** Synthesize Agent 1's patient data + Agent 2's guidelines into a coherent clinical analysis. This is the "brain" of MIRA.

In [18]:
def agent3_clinical_reasoning(state: MIRAState) -> MIRAState:
    print("\n" + "="*60)
    print("🤖 AGENT 3: Clinical Reasoning")
    print("="*60)

    sql_result   = state.get("sql_result", "No patient data available.")
    guidelines   = state.get("guidelines", "No guidelines found.")
    question     = state["clinical_question"]

    # Parse guidelines for cleaner formatting
    try:
        guideline_text = ""
        parsed_g = json.loads(guidelines)
        for g in parsed_g.get("guidelines", []):
            guideline_text += f"\n[{g['source']}] {g['topic']}:\n{g['text']}\n"
    except:
        guideline_text = guidelines

    system_prompt = """You are an expert clinical AI assistant.
You have been given:
1. Live patient data from the hospital database (SQL results)
2. Relevant clinical guidelines from the medical knowledge base

Your job: Synthesize both into a clear, structured clinical analysis.

Format your response as:
## Patient Summary
[Key patient demographics and findings from the database]

## Identified Concerns
[Specific abnormal values and what they indicate]

## Clinical Guideline Context
[What the guidelines say about these findings]

## Recommended Actions
[Evidence-based recommendations from the guidelines]

Always ground every claim in either the patient data or a cited guideline. Never hallucinate values."""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
Clinical Question: {question}

=== PATIENT DATA (from SQLite) ===
{sql_result[:2000]}

=== MEDICAL GUIDELINES (from FAISS) ===
{guideline_text[:2000]}

Provide your clinical analysis:
""")
    ])

    reasoning = response.content.strip()
    print(f"✅ Clinical reasoning generated ({len(reasoning)} chars)")
    print(f"\nPreview:\n{reasoning[:400]}...")

    return {
        **state,
        "clinical_reasoning": reasoning
    }


print("✅ Agent 3 defined")

✅ Agent 3 defined



**Job:** : Validates Agent 3's reasoning for:
- Hallucinated values (claims not grounded in DB or guidelines)
- Dangerous recommendations
- Missing citations

In [19]:
def agent4_critic_safety(state: MIRAState) -> MIRAState:
    print("\n" + "="*60)
    print("🤖 AGENT 4: Critic & Safety")
    print("="*60)

    reasoning  = state.get("clinical_reasoning", "")
    sql_result = state.get("sql_result", "")
    guidelines = state.get("guidelines", "")

    system_prompt = """You are a medical AI safety critic. Review the clinical analysis for:

1. HALLUCINATION CHECK: Are all patient values present in the SQL data? Flag any invented numbers.
2. GUIDELINE GROUNDING: Are recommendations backed by the provided guidelines?
3. SAFETY CHECK: Are there any dangerous or contraindicated recommendations?
4. COMPLETENESS: Does the analysis address the original question?

Respond in this exact JSON format:
{
  "approved": true/false,
  "safety_flags": ["flag1", "flag2"],
  "corrections": "any corrections needed or empty string",
  "final_report": "the approved or corrected clinical analysis text"
}

If approved, copy the analysis into final_report. If not, write a corrected version.
Return ONLY the JSON object, no markdown."""

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
=== ORIGINAL PATIENT DATA ===
{sql_result[:1500]}

=== GUIDELINES USED ===
{guidelines[:1000]}

=== CLINICAL ANALYSIS TO REVIEW ===
{reasoning}

Review and respond in JSON:
""")
    ])

    # Parse the critic's JSON response
    try:
        raw = response.content.strip().replace("```json", "").replace("```", "").strip()
        critic_output = json.loads(raw)
    except Exception as e:
        print(f"⚠️  Could not parse critic JSON: {e}")
        critic_output = {
            "approved": True,
            "safety_flags": [],
            "corrections": "",
            "final_report": reasoning
        }

    approved     = critic_output.get("approved", True)
    safety_flags = critic_output.get("safety_flags", [])
    final_report = critic_output.get("final_report", reasoning)

    print(f"{'✅ APPROVED' if approved else '❌ NOT APPROVED'}")
    if safety_flags:
        print(f"⚠️  Safety Flags: {safety_flags}")
    else:
        print("   No safety flags raised")

    return {
        **state,
        "final_report": final_report,
        "safety_flags": safety_flags,
        "approved": approved
    }


print("✅ Agent 4 defined")

✅ Agent 4 defined


In [20]:
# Build the graph
graph_builder = StateGraph(MIRAState)

# ── Add nodes (each agent is one node) 
graph_builder.add_node("sql_extractor",    agent1_sql_extractor)
graph_builder.add_node("semantic_crossref", agent2_semantic_crossref)
graph_builder.add_node("clinical_reasoning", agent3_clinical_reasoning)
graph_builder.add_node("critic_safety",    agent4_critic_safety)

# ── Entry point 
graph_builder.set_entry_point("sql_extractor")

# ── Conditional edge after Agent 1 (retry on SQL error) 
graph_builder.add_conditional_edges(
    "sql_extractor",
    should_retry_sql,
    {
        "retry": "sql_extractor",       # loop back to Agent 1
        "ok"   : "semantic_crossref"    # proceed to Agent 2
    }
)

# ── Linear edges for Agents 2 → 3 → 4 → END 
graph_builder.add_edge("semantic_crossref",   "clinical_reasoning")
graph_builder.add_edge("clinical_reasoning",  "critic_safety")
graph_builder.add_edge("critic_safety",       END)

# ── Compile 
mira_graph = graph_builder.compile()

print("✅ MIRA LangGraph compiled")
print("\nGraph flow:")
print("  START → sql_extractor → [retry?] → semantic_crossref → clinical_reasoning → critic_safety → END")

✅ MIRA LangGraph compiled

Graph flow:
  START → sql_extractor → [retry?] → semantic_crossref → clinical_reasoning → critic_safety → END


In [21]:
def run_mira(clinical_question: str) -> dict:
    """Run the full MIRA pipeline for a clinical question."""
    print("\n" + "🏥"*30)
    print(f"MIRA CLINICAL QUERY: {clinical_question}")
    print("🏥"*30)

    initial_state: MIRAState = {
        "clinical_question": clinical_question,
        "sql_query_used"   : "",
        "sql_result"       : "",
        "sql_retry_count"  : 0,
        "sql_error"        : "",
        "search_query_used": "",
        "guidelines"       : "",
        "clinical_reasoning": "",
        "final_report"     : "",
        "safety_flags"     : [],
        "approved"         : False
    }

    final_state = mira_graph.invoke(initial_state)
    return final_state


def print_mira_report(state: dict):
    """Pretty print the final MIRA output."""
    print("\n" + "="*60)
    print("📋 MIRA FINAL REPORT")
    print("="*60)
    print(f"\n🔎 SQL Used:\n{state.get('sql_query_used', 'N/A')}")
    print(f"\n🔍 Semantic Query: {state.get('search_query_used', 'N/A')}")
    print(f"\n{'✅ APPROVED' if state.get('approved') else '⚠️ FLAGGED'}")
    if state.get('safety_flags'):
        print(f"Safety Flags: {state['safety_flags']}")
    print(f"\n{'='*60}")
    print(state.get("final_report", "No report generated."))
    print(f"{'='*60}")


print("✅ run_mira() helper ready")

✅ run_mira() helper ready


In [22]:
state = run_mira(
    "Which patients have abnormal lab results? Summarize their key findings and any clinical concerns."
)
print_mira_report(state)


🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥
MIRA CLINICAL QUERY: Which patients have abnormal lab results? Summarize their key findings and any clinical concerns.
🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥

🤖 AGENT 1: SQL Data Extractor
📝 SQL Generated:
SELECT DISTINCT p.subject_id, p.gender, p.anchor_age, p.anchor_year, p.anchor_year_group, a.admission_type, a.admission_location, a.discharge_location, a.insurance, a.language, a.marital_status, a.race, d.label AS lab_test, l.value AS lab_value, l.valuenum AS lab_valuenum, l.valueuom AS lab_valueuom, l.flag AS lab_flag
FROM patients p
JOIN admissions a ON p.subject_id = a.subject_id
JOIN labevents l ON p.subject_id = l.subject_id
JOIN d_labitems d ON l.itemid = d.itemid
WHERE l.flag = 'abnormal';
✅ SQL Success: 20 rows returned

🤖 AGENT 2: Semantic Cross-Ref
🔍 Semantic Query: "clinical guidelines for low red blood cell count in adults, specifically addressing abnormal values like 3.35 m/uL"
✅ Found 3 guideline chunks
   #1 [Surviving Sepsis Campaign 2021] — Se

In [23]:
state = run_mira(
    "Find patients with abnormal creatinine values. Based on the guidelines, do any of them meet criteria for Acute Kidney Injury?"
)
print_mira_report(state)


🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥
MIRA CLINICAL QUERY: Find patients with abnormal creatinine values. Based on the guidelines, do any of them meet criteria for Acute Kidney Injury?
🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥🏥

🤖 AGENT 1: SQL Data Extractor
📝 SQL Generated:
SELECT DISTINCT p.subject_id
FROM patients p
JOIN admissions a ON p.subject_id = a.subject_id
JOIN labevents l ON a.hadm_id = l.hadm_id
JOIN d_labitems d ON l.itemid = d.itemid
WHERE d.label = 'Creatinine' AND l.flag = 'abnormal';
✅ SQL Success: 20 rows returned

🤖 AGENT 2: Semantic Cross-Ref
🔍 Semantic Query: "Acute Kidney Injury guidelines creatinine criteria abnormal values"
✅ Found 3 guideline chunks
   #1 [KDIGO AKI Guidelines 2012 (updated 2024)] — AKI — Definition and Staging
   #2 [KDIGO AKI Guidelines 2012 (updated 2024)] — AKI — Management
   #3 [Surviving Sepsis Campaign 2021] — Sepsis — Lab Indicators

🤖 AGENT 3: Clinical Reasoning
✅ Clinical reasoning generated (2091 chars)

Preview:
## Patient Summary
The patient data 

In [24]:
print("📊 FULL MIRA STATE BREAKDOWN")
print("="*60)

print(f"\n[Agent 1 — SQL Extractor]")
print(f"  SQL used     : {state['sql_query_used'][:200]}")
print(f"  Retry count  : {state['sql_retry_count']}")
print(f"  Error        : {state['sql_error'] or 'None'}")
print(f"  Rows returned: {len(json.loads(state['sql_result']).get('rows', [])) if state['sql_result'] else 0}")

print(f"\n[Agent 2 — Semantic Cross-Ref]")
print(f"  Search query : {state['search_query_used']}")
guidelines_count = len(json.loads(state['guidelines']).get('guidelines', [])) if state['guidelines'] else 0
print(f"  Guidelines   : {guidelines_count} chunks retrieved")

print(f"\n[Agent 3 — Clinical Reasoning]")
print(f"  Reasoning len: {len(state['clinical_reasoning'])} chars")

print(f"\n[Agent 4 — Critic & Safety]")
print(f"  Approved     : {state['approved']}")
print(f"  Safety flags : {state['safety_flags'] or 'None'}")
print(f"  Report len   : {len(state['final_report'])} chars")

📊 FULL MIRA STATE BREAKDOWN

[Agent 1 — SQL Extractor]
  SQL used     : SELECT DISTINCT p.subject_id
FROM patients p
JOIN admissions a ON p.subject_id = a.subject_id
JOIN labevents l ON a.hadm_id = l.hadm_id
JOIN d_labitems d ON l.itemid = d.itemid
WHERE d.label = 'Creati
  Retry count  : 0
  Error        : None
  Rows returned: 20

[Agent 2 — Semantic Cross-Ref]
  Search query : "Acute Kidney Injury guidelines creatinine criteria abnormal values"
  Guidelines   : 3 chunks retrieved

[Agent 3 — Clinical Reasoning]
  Reasoning len: 2091 chars

[Agent 4 — Critic & Safety]
  Approved     : False
  Safety flags : ['HALLUCINATION CHECK: Missing specific creatinine values', 'COMPLETENESS: Lack of specific patient data analysis']
  Report len   : 2161 chars
